In [ ]:
Here's my proposed solution...we keep pizza the same, we add a field to toppings called "cauliflower_crust" - if there's no value in this field, our program will accept the field value from pizza and seek to use it, if the cauliflower_crust field has a value, we'll accept that as the alternate field name that will supercede the field value from pizza for that toppings row.  We'll carry this into our slide_driver file and into the creation of the slides itself.  Show me where we'd apply this fix in my codeset.  current codeset: # -----------------------------
# LIBRARIES
# -----------------------------
import os
import time
import shutil
from pathlib import Path
from itertools import product
import urllib.parse
import pandas as pd
import urllib3
import keyring

from tableau_api_lib import TableauServerConnection
from tableau_api_lib.utils import querying, flatten_dict_column
import tableauserverclient as TSC

from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.oxml.xmlchemy import OxmlElement
from pptx.oxml.ns import qn
import aspose.slides as slides
from aspose.slides import Portion, PortionFormat
from collections import OrderedDict
from PIL import Image, ImageStat
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from collections import defaultdict


# TABLEAU SERVER CLIENT LIBRARY (TSC) - CONFIG & AUTH
# --------------------------------------------------
SERVICE_NAME = "tableau_tsc"

# Retrieve secrets
pat_name = keyring.get_password(SERVICE_NAME, "pat_name")
pat_token = keyring.get_password(SERVICE_NAME, "pat_token")
site_id = keyring.get_password(SERVICE_NAME, "site_id")
server_url = keyring.get_password(SERVICE_NAME, "server_url")

# Auth object
tableau_auth = TSC.PersonalAccessTokenAuth(
    pat_name,
    pat_token,
    site_id
)

server = TSC.Server(server_url, use_server_version=True)
server.add_http_options({'verify': False})
server.auth.sign_in(tableau_auth)
print("Signed in to Tableau Server successfully")

# -----------------------------
# FUNCTIONS
# -----------------------------
def build_titles(row):
    title = row["pizza_label"]

    subtitle_parts = [
        row["section"],
        row["toppings_label"]
    ]

    subtitle = " | ".join([p for p in subtitle_parts if pd.notna(p)])

    return title, subtitle

def clean_split(val):
    if pd.isna(val):
        return []
    return [
        v.strip().strip('"')
        for v in str(val).split("|")
        if v.strip()
    ]

def build_filter(field_name, values):
    field_encoded = urllib.parse.quote(field_name)
    encoded_values = [urllib.parse.quote(v) for v in values]
    return f"vf_{field_encoded}={','.join(encoded_values)}"

def build_filter_string(row, param_dict):
    filters = []

    # Pizza
    if pd.notna(row["pizza_field"]) and pd.notna(row["pizza_values"]):
        vals = clean_split(row["pizza_values"])
        filters.append(build_filter(row["pizza_field"], vals))

    # Toppings
    if pd.notna(row["toppings_field"]) and pd.notna(row["toppings_values"]):
        vals = clean_split(row["toppings_values"])
        filters.append(build_filter(row["toppings_field"], vals))

    # Extras (params)
    for k, v in param_dict.items():
        filters.append(
            f"vf_{urllib.parse.quote(k)}={urllib.parse.quote(str(v))}"
        )

    return "&".join(filters)   

def delete_workbooks_in_project(project_id):
    page_number = 1
    while True:
        req_options = TSC.RequestOptions(pagenumber=page_number)
        workbooks, pagination = server.workbooks.get(req_options=req_options)
        for wb in workbooks:
            if wb.project_id == project_id:
                print(f"Deleting workbook: {wb.name} ({wb.id})")
                server.workbooks.delete(wb.id)
        if len(workbooks) < pagination.page_size:
            break
        page_number += 1

def find_project_by_name(name):
    page_number = 1
    while True:
        req_options = TSC.RequestOptions(pagenumber=page_number)
        projects, pagination = server.projects.get(req_options=req_options)
        for proj in projects:
            if proj.name == name:
                return proj
        if len(projects) < pagination.page_size:
            break
        page_number += 1
    return None

def process_slide(row, deck_id):

    slide_id = row["slide_id"]
    story_name = row["story"]
    tab_path = row["tab_path"]

    wb_info = workbook_lookup.get(tab_path)
    if not wb_info:
        return None

    workbook_id = wb_info["workbook_id"]
    workbook_name = wb_info["workbook_name"]

    story_key = story_name.strip().lower()
    view_id = view_lookup.get(workbook_id, {}).get(story_key)

    if not view_id:
        return None

    param_dict = param_lookup.get(slide_id, {})
    filter_string = build_filter_string(row, param_dict)

    try:
        response = connection.query_view_image(
            view_id=view_id,
            parameter_dict={"filter": filter_string}
        )
    except Exception as e:
        log(f"❌ API error: {slide_id} | {e}")
        return None

    safe_slide = (
        slide_id.replace(" | ", "_")
        .replace("/", "_")
        .replace(":", "")
    )

    file_name = f"{deck_id}_{safe_slide}.png"
    file_path = output_dir / file_name

    with open(file_path, "wb") as f:
        f.write(response.content)

    # --- Validate image ---
    file_size_kb = os.path.getsize(file_path) / 1024
    is_small = file_size_kb < 40

    try:
        with Image.open(file_path).convert("L") as img:
            stat = ImageStat.Stat(img)
            is_blank = stat.stddev[0] < 5
    except:
        is_blank = True

    param_lines = [
        f"{row['pizza_field']}: {row['pizza_values']}",
        f"{row['toppings_field']}: {row['toppings_values']}",
    ]

    for k, v in param_dict.items():
        param_lines.append(f"{k}: {v}")

    param_text = "Report Parameters:\n" + "\n".join(param_lines)
    title, subtitle = build_titles(row)

    if is_small or is_blank:
        try:
            os.remove(file_path)
        except:
            pass

        return {
            "image_path": None,
            "param_text": param_text + "\n\nNot Applicable",
            "ppt_section": row["section"],
            "ppt_title": title,
            "ppt_subtitle": subtitle,
            "deck_id": deck_id,   
            "is_empty": True
        }

    return {
        "image_path": file_path,
        "param_text": param_text,
        "ppt_section": row["section"],
        "ppt_title": title,
        "ppt_subtitle": subtitle,
        "deck_id": deck_id,   
        "is_empty": False
    }


# -----------------------------
# DRIVERS
# -----------------------------
# Load data
pizza = pd.read_csv(r"C:\Users\medwards\OneDrive - HDR, Inc\sandbox\tableau-rpa\code\pizza.csv")
toppings = pd.read_csv(r"C:\Users\medwards\OneDrive - HDR, Inc\sandbox\tableau-rpa\code\toppings.csv")

# Normalize
pizza.columns = pizza.columns.str.strip().str.lower()
toppings.columns = toppings.columns.str.strip().str.lower()

# Rename for clarity
pizza = pizza.rename(columns={
    "field": "pizza_field",
    "operation": "pizza_operation",
    "values": "pizza_values",
    "label": "pizza_label"
})

toppings = toppings.rename(columns={
    "field": "toppings_field",
    "operation": "toppings_operation",
    "values": "toppings_values",
    "label": "toppings_label"
})

# CROSS JOIN (deck × slides)
pizza["key"] = 1
toppings["key"] = 1

driver = pizza.merge(toppings, on="key").drop(columns=["key"])

# Final structure
driver = driver[
    [
        "tab_path",
        "section",
        "story",
        "pizza_field", "pizza_operation", "pizza_values", "pizza_label",
        "toppings_field", "toppings_operation", "toppings_values", "toppings_label",
        "extras","extra_default_values"
    ]
]

# IDs
driver["deck_id"] = driver["pizza_label"]
driver["slide_id"] = (
    driver["pizza_label"] + " | " +
    driver["section"] + " | " +
    driver["toppings_label"]
)

# -----------------------------
# BUILD PARAM TABLE 
# -----------------------------
# Normalize extras + defaults
driver["extras"] = driver["extras"].fillna("").astype(str)
driver["extra_default_values"] = driver["extra_default_values"].fillna("").astype(str)

# Split both into lists
params = driver[["slide_id", "extras", "extra_default_values"]].copy()

params["extras"] = params["extras"].str.split("|")
params["extra_default_values"] = params["extra_default_values"].str.split("|")

# Explode BOTH columns together (keeps alignment)
params = params.explode(["extras", "extra_default_values"])

# Clean values (strip whitespace + quotes)
params["param_name"] = params["extras"].str.strip().str.strip('"')
params["param_value"] = params["extra_default_values"].str.strip().str.strip('"')

# Drop originals
params = params.drop(columns=["extras", "extra_default_values"])

# Remove blanks
params = params[params["param_name"] != ""]

# Reset index
params = params.reset_index(drop=True)

# -----------------------------
# DISABLE WARNINGS
# -----------------------------
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# -----------------------------
# PATHS + RUN STRUCTURE
# -----------------------------
code_dir = Path.cwd()
project_root = code_dir.parent

data_dir = project_root / "data"
runs_dir = data_dir / "runs"

timestamp = time.strftime("%Y%m%d_%H%M%S")

run_dir = runs_dir / f"run_{timestamp}"
output_dir = run_dir / "outputs"
code_archive_dir = run_dir / "code"

for d in [run_dir, output_dir, code_archive_dir]:
    d.mkdir(parents=True, exist_ok=True)

log_file = run_dir / "logfile.txt"

def log(msg):
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(line + "\n")

# Save script snapshot
try:
    shutil.copy(Path.cwd() / "ppt_creator.ipynb", code_archive_dir / "ppt_creator.ipynb")
except Exception:
    pass

pptx_template_path = Path(r"C:\Users\medwards\OneDrive - HDR, Inc\sandbox\tableau-rpa\data\input\20260615_HOSPITAL_TEMPLATE_CurrentState.pptx")
pptx_output_path = output_dir / f"{pptx_template_path.stem}_insert{pptx_template_path.suffix}"

log("🚀 Pipeline started")
log(f"Run directory: {run_dir}")

# -----------------------------
# CLEAN DRIVER OUTPUT
# -----------------------------
# Drop extras (no longer needed)
driver = driver.drop(columns=["extras"])

# -----------------------------
# SAVE FILES
# -----------------------------
driver.to_csv("slide_driver.csv", index=False)
params.to_csv("slide_params.csv", index=False)

print(f"Generated {len(driver)} slides.")
print(f"Generated {len(params)} parameters.")

# -----------------------------
# TABLEAU PROJECT INIT
# -----------------------------
project_name = "Tableau_RPA"
# --- Find project by paging through results ---
existing_project = find_project_by_name(project_name)
if existing_project:
    project_id = existing_project.id
    print(f"Using existing project: {project_name} ({project_id})")
else:
    new_project_item = TSC.ProjectItem(
        name=project_name,
        content_permissions='LockedToProject',
        description='Temp folder for Tableau RPA exports'
    )
    new_project = server.projects.create(new_project_item)
    project_id = new_project.id
    print(f"Created project: {project_name} ({project_id})")
# --- Delete existing workbooks in the project ---
delete_workbooks_in_project(project_id)

# -----------------------------
# PUBLICATION
# -----------------------------
unique_tab_paths = driver["tab_path"].unique()
workbook_lookup = {}

for tab_path in unique_tab_paths:

    log(f"Publishing workbook: {tab_path}")

    workbook_path = Path(tab_path)
    workbook_name = workbook_path.stem

    new_workbook = TSC.WorkbookItem(
        name=workbook_name,
        project_id=project_id
    )

    published = server.workbooks.publish(
        new_workbook,
        tab_path,
        'CreateNew',
        skip_connection_check=True
    )

    workbook_lookup[tab_path] = {
        "workbook_id": published.id,
        "workbook_name": workbook_name
    }

    log(f"Published: {workbook_name} ({published.id})")

# --------------------------------------------------
# TABLEAU API LIBRARY (TAL) - CONFIG & AUTH
# --------------------------------------------------
SERVICE_NAME = "tableau_tal"   
# Retrieve secrets from keyring
pat_name = keyring.get_password(SERVICE_NAME, "pat_name")
pat_token = keyring.get_password(SERVICE_NAME, "pat_token")
site_name = keyring.get_password(SERVICE_NAME, "site_name")
server_url = keyring.get_password(SERVICE_NAME, "server_url")
tableau_config = {
    'tableau_prod': {
        'server': server_url,
        'api_version': '3.27',
        'personal_access_token_name': pat_name,
        'personal_access_token_secret': pat_token,
        'site_name': site_name,
        'site_url': site_name,
        'cache_buster': '',
        'temp_dir': ''
    }
}

connection = TableauServerConnection(tableau_config, ssl_verify=False)
connection.sign_in()
print("Signed in to Tableau via TAL successfully")

view_df = querying.get_views_dataframe(connection)
views_with_wb_info_df = flatten_dict_column(
    df=view_df,
    keys=["name", "id"],
    col_name="workbook"
)
views_by_workbook = {
    wb_id: df for wb_id, df in views_with_wb_info_df.groupby("workbook_id")
}
view_lookup = {}
for wb_id, df in views_by_workbook.items():
    view_lookup[wb_id] = {
        row["name"].strip().lower(): row["id"]
        for _, row in df.iterrows()
        if row["sheetType"].lower() == "story"
    }

# -----------------------------
# DECK BUILD
# -----------------------------
# Build param lookup by slide_id
param_lookup = (
    params
    .groupby("slide_id")
    .apply(lambda df: dict(zip(df["param_name"], df["param_value"])))
    .to_dict()
)

slide_metadata = []
MAX_WORKERS = 6  

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = [
        executor.submit(process_slide, row, deck_id)
        for deck_id, deck_df in driver.groupby("deck_id")
        for _, row in deck_df.iterrows()
    ]

    for future in tqdm(as_completed(futures), total=len(futures), desc="Rendering slides"):
        result = future.result()
        if result:
            slide_metadata.append(result)

if not slide_metadata:
    log("No slides generated.")
    exit()

decks = defaultdict(list)

for item in slide_metadata:
    decks[item["deck_id"]].append(item)

for deck_id, items in decks.items():

    log(f"📊 Building deck: {deck_id}")

    pptx_output_path = output_dir / f"{deck_id}.pptx"

    sectioned = OrderedDict()

    for item in items:
        sec = str(item.get("ppt_section") or "Default Section")
        sectioned.setdefault(sec, []).append(item)

    with slides.Presentation(str(pptx_template_path)) as presentation:

        layout = next((l for l in presentation.layout_slides if l.name == "1_Text"), None)
        if not layout:
            raise ValueError("Layout '1_Text' not found")

        section_first_slide = {}

        for section_name, section_items in sectioned.items():

            for item in section_items:

                slide = presentation.slides.add_empty_slide(layout)

                if section_name not in section_first_slide:
                    section_first_slide[section_name] = slide

                # --- Title ---
                for shape in slide.shapes:
                    if shape.placeholder and shape.placeholder.type == slides.PlaceholderType.TITLE:
                        shape.text_frame.text = item["ppt_title"]

                # --- Subtitle ---
                subtitle_box = slide.shapes.add_auto_shape(
                    slides.ShapeType.RECTANGLE,
                    36, 80,
                    presentation.slide_size.size.width - 72,
                    40
                )

                tf = subtitle_box.text_frame
                p = tf.paragraphs[0]

                portion = Portion()
                portion.text = item.get("ppt_subtitle", "")
                portion.portion_format.font_height = 14
                p.portions.add(portion)

                # --- Param box ---
                textbox = slide.shapes.add_auto_shape(
                    slides.ShapeType.RECTANGLE,
                    519.12, 14.4, 425.52, 104.4
                )

                tf = textbox.text_frame
                p = tf.paragraphs[0]

                portion = Portion()
                portion.text = item["param_text"]
                portion.portion_format.font_height = 9
                p.portions.add(portion)

                # --- Image / Placeholder ---
                if item["is_empty"]:
                    tb = slide.shapes.add_auto_shape(
                        slides.ShapeType.RECTANGLE,
                        36, 120,
                        presentation.slide_size.size.width - 72,
                        300
                    )

                    tf = tb.text_frame
                    p = tf.paragraphs[0]
                    p.paragraph_format.alignment = slides.TextAlignment.CENTER

                    portion = Portion()
                    portion.text = "Not Applicable"
                    portion.portion_format.font_height = 24
                    p.portions.add(portion)

                else:
                    with open(item["image_path"], "rb") as img:
                        image = presentation.images.add_image(img)

                    slide.shapes.add_picture_frame(
                        slides.ShapeType.RECTANGLE,
                        36,
                        118.8,
                        presentation.slide_size.size.width - 72,
                        400,
                        image
                    )

        presentation.sections.clear()

        for section_name, first_slide in section_first_slide.items():
            presentation.sections.add_section(section_name, first_slide)

        presentation.save(str(pptx_output_path), slides.export.SaveFormat.PPTX)

    log(f"✅ Saved deck: {pptx_output_path}")

# WATERMARK CLEANUP
for deck_id in decks:

    pptx_output_path = output_dir / f"{deck_id}.pptx"

    prs = Presentation(str(pptx_output_path))

    for slide in prs.slides:
        shapes_to_remove = []

        for shape in slide.shapes:
            if shape.has_text_frame:
                text = shape.text_frame.text or ""
                if any(w in text.lower() for w in ["evaluation", "aspose", "copyright"]):
                    shapes_to_remove.append(shape)

        for shape in shapes_to_remove:
            sp = shape._element
            sp.getparent().remove(sp)

    prs.save(str(pptx_output_path))

    print(f"✅ Watermark removed: {pptx_output_path.name}")

Signed in to Tableau Server successfully
[2026-06-17 08:45:05] 🚀 Pipeline started
[2026-06-17 08:45:05] Run directory: c:\Users\medwards\OneDrive - HDR, Inc\sandbox\tableau-rpa\data\runs\run_20260617_084505
Generated 54 slides.
Generated 81 parameters.
Using existing project: Tableau_RPA (9f7f9086-7343-4955-b5d6-4fc8330d3e38)
Deleting workbook: inpatient_ECU (19bc0b02-50d1-4204-a2c3-bc2b18a5131b)
Deleting workbook: surgery_ECU (ca9ebf18-7e90-4ac0-a585-067070058008)
[2026-06-17 08:45:06] Publishing workbook: C:\Users\medwards\OneDrive - HDR, Inc\Arch. Advisory Services - Clients\N_Carolina\ECU\2024_MasterPlan\Analysis\Inpatient\inpatient_ECU.twbx
[2026-06-17 08:45:08] Published: inpatient_ECU (ec672e99-8506-4503-a925-e35cf6b62daa)
[2026-06-17 08:45:08] Publishing workbook: C:\Users\medwards\OneDrive - HDR, Inc\Arch. Advisory Services - Clients\N_Carolina\ECU\2024_MasterPlan\Analysis\Surgery\surgery_ECU.twbx
[2026-06-17 08:45:09] Published: surgery_ECU (5531969d-4bce-493a-8bf1-02570912a9

KeyboardInterrupt: 